# HumAID — Zero-shot Classification (Filtered Labels, Batch API, Sharding)

- **Filtered labels (per event):** prompts + JSON schema only list labels that appear in that event’s ground truth → reduces out-of-scope (OOS) predictions.
- **Batch API flow:** build `requests.jsonl` → upload → create batch → poll → download `outputs.jsonl` (and `errors.jsonl` if any).
- **Patch pass:** after batch completes, any missing/blank predictions are re-classified synchronously so `predictions.csv` has one row per input.
- **Stratified sharding (optional):** split large events into *k* shards **preserving class ratios**; use the **same** event-level labels + rules for all shards; merge predictions back in original order.
- **Reporting:** confusion matrices (counts + row-normalized), per-class F1/error, mistakes CSV, and a sortable `results/index.html`.  
  - **Scope** = label universe used for metrics (default `truth`).  
  - **OOS preds** = predictions not in the truth set (QA signal).

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [2]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_3

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_3
TAG = "modeS-gpt-4o-RULES3-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 90_000   # Tier-1 cap
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

# 1) Discover datasets (events/splits)

In [3]:
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run as single batch:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Will be sharded (exceeds cap):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])

,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
8,kaikoura_earthquake_2016,test,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,435,1036,450660,False,500.7
1,canada_wildfires_2016,test,Dataset\HumAID\canada_wildfires_2016\canada_wi...,445,1022,454790,False,505.3
2,cyclone_idai_2019,test,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,779,1068,831972,False,924.4
4,hurricane_florence_2018,test,Dataset\HumAID\hurricane_florence_2018\hurrica...,1241,1050,1303050,False,1447.8
7,hurricane_maria_2017,test,Dataset\HumAID\hurricane_maria_2017\hurricane_...,1442,1036,1493912,False,1659.9
0,california_wildfires_2018,test,Dataset\HumAID\california_wildfires_2018\calif...,1461,1056,1542816,False,1714.2
3,hurricane_dorian_2019,test,Dataset\HumAID\hurricane_dorian_2019\hurricane...,1508,1052,1586416,False,1762.7
9,kerala_floods_2018,test,Dataset\HumAID\kerala_floods_2018\kerala_flood...,1582,1057,1672174,False,1858.0
5,hurricane_harvey_2017,test,Dataset\HumAID\hurricane_harvey_2017\hurricane...,1805,1035,1868175,False,2075.8
6,hurricane_irma_2017,test,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,1862,1035,1927170,False,2141.3


OK to run as single batch:


,event,split,num_rows,est_total_tokens,limit_used_%


Will be sharded (exceeds cap):


,event,split,num_rows,est_total_tokens,limit_used_%
0,kaikoura_earthquake_2016,test,435,450660,500.7
1,canada_wildfires_2016,test,445,454790,505.3
2,cyclone_idai_2019,test,779,831972,924.4
3,hurricane_florence_2018,test,1241,1303050,1447.8
4,hurricane_maria_2017,test,1442,1493912,1659.9
5,california_wildfires_2018,test,1461,1542816,1714.2
6,hurricane_dorian_2019,test,1508,1586416,1762.7
7,kerala_floods_2018,test,1582,1672174,1858.0
8,hurricane_harvey_2017,test,1805,1868175,2075.8
9,hurricane_irma_2017,test,1862,1927170,2141.3


# 2) Run all datasets (sequentially)

In [4]:
def run_list_single(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that already fit under the cap using the normal runner."""
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running (single) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": "single",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": "single",
            })
    return pd.DataFrame(results)

def run_list_sharded(dflist: pd.DataFrame, token_df: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that exceed the cap using stratified shards. k is computed from token estimates."""
    results = []
    # Build a quick lookup: (event,split) -> est_total_tokens
    est_map = {(r.event, r.split): r.est_total_tokens for r in token_df.itertuples(index=False)}
    eff_cap = BATCH_TOKEN_LIMIT * SAFETY_MARGIN

    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        est_tokens = est_map.get((event, split), None)
        # Conservative shard count: ceil(est / eff_cap). Min 2.
        k = max(2, math.ceil((est_tokens or (eff_cap + 1)) / eff_cap))
        print(f"\n=== Running (sharded x{k}) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment_sharded(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=f"{tag}-sharded{k}",
                k_shards=k,
                temperature=0.0,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
                analysis_subdir="analysis",  # merged analysis
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": f"sharded{k}",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": f"sharded{k}",
            })
    return pd.DataFrame(results)

In [5]:
# --- Run singles with your normal key (optional context manager kept for parity)
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using OPENAI_API_KEY_1")
    df_runs_single = run_list_single(df_fit, RULES, MODEL, tag=f"{TAG}-TIER1")

# --- Run sharded for the too-big ones (same key or another if you prefer)
# You can keep the same key; sharding is already controlling token usage.
with use_api_key_env("OPENAI_API_KEY_1"):
    if not df_too_big.empty:
        df_runs_sharded = run_list_sharded(df_too_big, token_index, RULES, MODEL, tag=f"{TAG}")
    else:
        df_runs_sharded = pd.DataFrame()
        print("No large datasets to shard.")

# 3) Save a small index of all runs
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

all_runs = pd.concat([df_runs_single, df_runs_sharded], ignore_index=True)
all_runs.to_csv(idx_dir / f"runs_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run index at:", idx_dir)
display(all_runs)

>>> Using OPENAI_API_KEY_1

=== Running (sharded x6) kaikoura_earthquake_2016/test (gpt-4o | modeS-gpt-4o-RULES3-filtered) ===
[batch batch_6911902573ec8190b3147f50694a7a04] status = validating
[batch batch_6911902573ec8190b3147f50694a7a04] status = completed
[batch batch_69119154f2cc8190b0f1f8d336115c86] status = validating
[batch batch_69119154f2cc8190b0f1f8d336115c86] status = completed
[batch batch_69119284bc648190925bad99ab6c1263] status = validating
[batch batch_69119284bc648190925bad99ab6c1263] status = completed
[batch batch_691193b3f62481909a8b0edb36e1df6d] status = validating
[batch batch_691193b3f62481909a8b0edb36e1df6d] status = completed
[batch batch_691194e675e48190a904446fde4f7851] status = validating
[batch batch_691194e675e48190a904446fde4f7851] status = completed
[batch batch_69119614f2088190802ff7c971113ded] status = validating
[batch batch_69119614f2088190802ff7c971113ded] status = completed
Saved merged predictions to: runs\kaikoura_earthquake_2016\test\gpt-4o\2025

,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total,mode
0,kaikoura_earthquake_2016,test,runs\kaikoura_earthquake_2016\test\gpt-4o\2025...,runs\kaikoura_earthquake_2016\test\gpt-4o\2025...,0.734210,0.685057,435,sharded6
1,canada_wildfires_2016,test,runs\canada_wildfires_2016\test\gpt-4o\2025110...,runs\canada_wildfires_2016\test\gpt-4o\2025110...,0.643152,0.746067,445,sharded6
2,cyclone_idai_2019,test,runs\cyclone_idai_2019\test\gpt-4o\20251110-00...,runs\cyclone_idai_2019\test\gpt-4o\20251110-00...,0.618270,0.736842,779,sharded11
3,hurricane_florence_2018,test,runs\hurricane_florence_2018\test\gpt-4o\20251...,runs\hurricane_florence_2018\test\gpt-4o\20251...,0.665919,0.726027,1241,sharded17
4,hurricane_maria_2017,test,runs\hurricane_maria_2017\test\gpt-4o\20251110...,runs\hurricane_maria_2017\test\gpt-4o\20251110...,0.607049,0.634976,1442,sharded19
5,california_wildfires_2018,test,ERROR,,NaN,NaN,0,sharded20
6,hurricane_dorian_2019,test,ERROR,,NaN,NaN,0,sharded20
7,kerala_floods_2018,test,runs\kerala_floods_2018\test\gpt-4o\20251110-0...,runs\kerala_floods_2018\test\gpt-4o\20251110-0...,0.571533,0.703540,1582,sharded21
8,hurricane_harvey_2017,test,ERROR,,NaN,NaN,0,sharded24
9,hurricane_irma_2017,test,ERROR,,NaN,NaN,0,sharded24


# Other experiments

In [4]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_3

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_3
TAG = "modeS-gpt-4o-RULES3-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 90_000   # Tier-1 cap
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

K = 6  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "kaikoura_earthquake_2016" / "kaikoura_earthquake_2016_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary



[batch batch_691187ba9ea88190aed4abcb9fe8bac7] status = validating
[batch batch_691187ba9ea88190aed4abcb9fe8bac7] status = completed
[batch batch_691188e9c8488190a87a8c707c7148b6] status = validating
[batch batch_691188e9c8488190a87a8c707c7148b6] status = completed
[batch batch_69118a1868988190a7bf3708bee65d8e] status = validating
[batch batch_69118a1868988190a7bf3708bee65d8e] status = completed
[batch batch_69118b477e088190b51646f2807cfbc7] status = validating
[batch batch_69118b477e088190b51646f2807cfbc7] status = completed
[batch batch_69118c7809788190bd5b84e88f3281c7] status = validating
[batch batch_69118c7809788190bd5b84e88f3281c7] status = completed
[batch batch_69118da64cc48190aa5bf453c214fedc] status = validating
[batch batch_69118da64cc48190aa5bf453c214fedc] status = completed
Saved merged predictions to: runs\kaikoura_earthquake_2016\test\gpt-4o\20251109-223536-modeS-gpt-4o-RULES3-filtered-sharded6\predictions.csv
Macro-F1 (merged): 0.736506362349943


{'num_total_with_truth': 435,
 'num_correct': 299,
 'num_incorrect': 136,
 'accuracy': 0.6873563218390805,
 'macro_f1': 0.736506362349943,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}

In [3]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 90_000   # Tier-1 cap
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

K = 3  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "kaikoura_earthquake_2016" / "kaikoura_earthquake_2016_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary

[batch batch_6911812630f48190bc3e8df93d3ce65b] status = validating
[batch batch_6911812630f48190bc3e8df93d3ce65b] status = completed
[batch batch_691182555fc08190bb74c670cf312c26] status = validating
[batch batch_691182555fc08190bb74c670cf312c26] status = completed
[batch batch_69118383ef4c8190b35170fbf19deba6] status = validating
[batch batch_69118383ef4c8190b35170fbf19deba6] status = completed
Saved merged predictions to: runs\kaikoura_earthquake_2016\test\gpt-4o\20251109-220732-modeS-gpt-4o-RULES1-filtered-sharded3\predictions.csv
Macro-F1 (merged): 0.7426308873939477


{'num_total_with_truth': 435,
 'num_correct': 324,
 'num_incorrect': 111,
 'accuracy': 0.7448275862068966,
 'macro_f1': 0.7426308873939477,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}